In [1]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
from scipy.linalg import sqrtm

# -------------------------- 参数配置（已修改路径） --------------------------
REAL_IMAGE_DIR = "../dataset/groundtruth"  # 真实图像文件夹（你的路径）
GEN_IMAGE_DIR = "../FlexIcon/test_output/batch"  # 生成图像文件夹（你的路径）
IMAGE_SIZE = 299  # Inception-v3要求的输入尺寸（300x300图像会被缩放到299x299）
BATCH_SIZE = 32  # 可根据显存调整（如遇显存不足，改小为16或8）
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {DEVICE}")

# -------------------------- 数据集类（适配新路径） --------------------------
class ImageDataset(Dataset):
    def __init__(self, image_dir, transform):
        # 读取指定路径下的所有图像（支持png、jpg、jpeg）
        self.image_paths = [
            os.path.join(image_dir, f)
            for f in os.listdir(image_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
        # 检查路径是否有效（若为空，提示错误）
        assert len(self.image_paths) > 0, f"路径 {image_dir} 中未找到图像文件！请检查路径是否正确。"
        self.transform = transform
        print(f"从 {image_dir} 加载图像 {len(self.image_paths)} 张，将缩放至299x299")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")  # 确保图像为3通道RGB
        return self.transform(img)  # 应用预处理（缩放至299x299）

# -------------------------- 特征提取函数（修正维度问题） --------------------------
def extract_features(loader, model, device):
    model.eval()  # 评估模式
    features = []

    with torch.no_grad():  # 关闭梯度计算，节省显存
        for imgs in loader:
            imgs = imgs.to(device)  # 输入形状：[batch_size, 3, 299, 299]（4维，符合模型要求）
            feat = model(imgs)  # 模型输出卷积层特征：[batch_size, 2048, 1, 1]（4维）
            feat = feat.view(feat.size(0), -1)  # 展平为2048维：[batch_size, 2048]
            features.append(feat.cpu().numpy())  # 转移到CPU并保存

    return np.concatenate(features, axis=0)  # 合并所有特征：[总样本数, 2048]

# -------------------------- FID计算函数 --------------------------
def calculate_fid(real_features, gen_features):
    # 计算均值和协方差
    mu_real = np.mean(real_features, axis=0)
    mu_gen = np.mean(gen_features, axis=0)
    sigma_real = np.cov(real_features, rowvar=False)
    sigma_gen = np.cov(gen_features, rowvar=False)

    # 计算FID核心公式
    mean_diff = np.sum((mu_real - mu_gen) **2)  # 均值差的平方和
    covmean = sqrtm(sigma_real.dot(sigma_gen))  # 协方差矩阵的平方根
    if np.iscomplexobj(covmean):
        covmean = covmean.real  # 移除数值误差导致的虚部
    cov_term = np.trace(sigma_real + sigma_gen - 2 * covmean)  # 协方差项

    return mean_diff + cov_term  # 总FID值

# -------------------------- 主运行逻辑 --------------------------
if __name__ == "__main__":
    # 图像预处理：300x300 → 299x299（匹配Inception-v3输入）
    transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),  # 缩放至299x299
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],  # ImageNet标准化参数（必须与预训练模型一致）
            std=[0.229, 0.224, 0.225]
        )
    ])

    # 加载真实图像和生成图像数据集
    real_dataset = ImageDataset(REAL_IMAGE_DIR, transform)
    gen_dataset = ImageDataset(GEN_IMAGE_DIR, transform)
    real_loader = DataLoader(real_dataset, batch_size=BATCH_SIZE, shuffle=False)
    gen_loader = DataLoader(gen_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # 加载Inception-v3模型（修正结构，避免维度错误）
    inception_model = models.inception_v3(pretrained=True, progress=False)
    inception_model.fc = nn.Identity()  # 移除全连接层，保留卷积层输出的2048维特征
    inception_model.to(DEVICE)

    # 提取特征
    print("提取真实图像特征...")
    real_feat = extract_features(real_loader, inception_model, DEVICE)
    print("提取生成图像特征...")
    gen_feat = extract_features(gen_loader, inception_model, DEVICE)

    # 计算并输出FID
    fid_score = calculate_fid(real_feat, gen_feat)
    print(f"\nFréchet Inception Distance: {fid_score:.4f}")

使用设备: cuda
从 ../dataset/groundtruth 加载图像 10076 张，将缩放至299x299
从 ../FlexIcon/test_output/batch 加载图像 10076 张，将缩放至299x299


C:\Users\35278\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\35278\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


提取真实图像特征...
提取生成图像特征...

Fréchet Inception Distance: 12.3338
